In [ ]:
import contextlib

import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from jaxtyping import Float
from plotly.subplots import make_subplots
from torch import Tensor
from tqdm.autonotebook import tqdm

from src.config.base import BaseConfig
from src.model.mlp import StackedResidualMLPConfig
from src.model.time_conditioning import TimeConditioningConfig

DEVICE = (
    "cuda:1"
    if torch.cuda.is_available() and torch.cuda.device_count() > 1
    else ("cuda:0" if torch.cuda.is_available() else "cpu")
)
USE_CUDA = DEVICE.startswith("cuda")
AMP_DTYPE = torch.bfloat16

torch.set_float32_matmul_precision("high")


@contextlib.contextmanager
def training_device_context():
    if USE_CUDA:
        with (
            torch.cuda.device(DEVICE),
            torch.device(DEVICE),
            torch.autocast(device_type="cuda", dtype=AMP_DTYPE),
        ):
            yield
        return
    with torch.device(DEVICE):
        yield


print(f"device: {DEVICE}")


In [ ]:
class BimodalDataConfig(BaseConfig):
    offset: Float[Tensor, "2"]
    center: float
    std: float
    scale: float
    std_cap: float = 2

    def _sample_capped_noise(
        self,
        batch_size: int,
    ) -> Float[Tensor, "batch 2"]:
        z = torch.randn((batch_size, 2))
        norms = z.norm(dim=1, keepdim=True).clamp_min(1e-6)
        capped_norms = norms.clamp_max(self.std_cap)
        return z * (self.std * capped_norms / norms)

    def sample_mode(
        self,
        batch_size: int,
        mode_id: int,
    ) -> Float[Tensor, "batch 2"]:
        x = self._sample_capped_noise(batch_size)
        direction = 1.0 if mode_id == 0 else -1.0
        x[:, 0] += direction * self.center / self.scale
        return x * self.scale + self.offset

    def sample_with_mode_ids(
        self,
        batch_size: int,
    ) -> tuple[Float[Tensor, "batch 2"], Tensor]:
        if batch_size % 2 != 0:
            raise ValueError("batch_size must be even")
        per_mode_size = batch_size // 2
        mode_ids = torch.cat(
            [
                torch.zeros(per_mode_size, dtype=torch.long),
                torch.ones(per_mode_size, dtype=torch.long),
            ]
        )
        x = torch.cat(
            [
                self.sample_mode(per_mode_size, mode_id=0),
                self.sample_mode(per_mode_size, mode_id=1),
            ],
            dim=0,
        )
        return x, mode_ids

    def sample(self, batch_size: int) -> Float[Tensor, "batch 2"]:
        x, _ = self.sample_with_mode_ids(batch_size)
        return x

    def sample_prior(self, batch_size: int) -> Float[Tensor, "batch 2"]:
        return torch.randn((batch_size, 2))

    def sample_data_mode_contours(
        self,
        per_contour_size: int,
        zscores: tuple[float, ...],
    ) -> tuple[Float[Tensor, "batch 2"], Tensor, Tensor, tuple[float, ...]]:
        angles = torch.linspace(0.0, 2.0 * torch.pi, per_contour_size + 1)[:-1]
        unit_circle = torch.stack([torch.cos(angles), torch.sin(angles)], dim=-1)
        contour_points = []
        contour_ids = []
        mode_ids = []
        contour_scale = self.scale * self.std
        for mode_id in [0, 1]:
            direction = 1.0 if mode_id == 0 else -1.0
            mode_center = torch.tensor(
                [direction * self.center / self.scale, 0.0],
                dtype=self.offset.dtype,
            )
            mode_center = mode_center * self.scale + self.offset
            for contour_id, zscore in enumerate(zscores):
                contour_points.append(mode_center + contour_scale * zscore * unit_circle)
                contour_ids.append(
                    torch.full((per_contour_size,), contour_id, dtype=torch.long)
                )
                mode_ids.append(
                    torch.full((per_contour_size,), mode_id, dtype=torch.long)
                )
        return (
            torch.cat(contour_points, dim=0),
            torch.cat(contour_ids, dim=0),
            torch.cat(mode_ids, dim=0),
            zscores,
        )

    def sample_latent_contours(
        self,
        per_contour_size: int,
        zscores: tuple[float, ...],
    ) -> tuple[Float[Tensor, "batch 2"], Tensor, tuple[float, ...]]:
        angles = torch.linspace(0.0, 2.0 * torch.pi, per_contour_size + 1)[:-1]
        unit_circle = torch.stack([torch.cos(angles), torch.sin(angles)], dim=-1)
        contour_points = []
        contour_ids = []
        for contour_id, zscore in enumerate(zscores):
            contour_points.append(zscore * unit_circle)
            contour_ids.append(
                torch.full((per_contour_size,), contour_id, dtype=torch.long)
            )
        return torch.cat(contour_points, dim=0), torch.cat(contour_ids, dim=0), zscores


class FlowMatchingExperimentConfig(BaseConfig):
    train_batch_size: int
    total_steps: int
    lr: float
    weight_decay: float
    grad_clip_norm: float
    log_every_steps: int
    snapshot_every_steps: int
    scatter_num_samples: int
    contour_points_per_level: int
    integration_steps: int
    final_toggle_steps: int
    contour_levels: tuple[float, ...]
    inverse_dataset_size: int
    inverse_batch_size: int
    inverse_steps: int
    inverse_lr: float
    inverse_weight_decay: float
    inverse_log_every_steps: int


data_config = BimodalDataConfig(
    offset=torch.tensor([0.0, 0.0]),
    center=3.0,
    std=0.2,
    scale=2.0,
)

experiment_config = FlowMatchingExperimentConfig(
    train_batch_size=2048,
    total_steps=2000,
    lr=3e-4,
    weight_decay=1e-4,
    grad_clip_norm=1.0,
    log_every_steps=50,
    snapshot_every_steps=500,
    scatter_num_samples=4096,
    contour_points_per_level=160,
    integration_steps=80,
    final_toggle_steps=11,
    contour_levels=(0.1, 0.5, 1.0, 1.5),
    inverse_dataset_size=65536,
    inverse_batch_size=2048,
    inverse_steps=2000,
    inverse_lr=3e-4,
    inverse_weight_decay=1e-4,
    inverse_log_every_steps=50,
)

experiment_config


In [ ]:
class FlowMatchingModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.velocity = StackedResidualMLPConfig.initialize(
            layer_dims=[2, 512, 512, 512, 512, 2],
            time_conditioning_config=TimeConditioningConfig(
                min_t_lambda=5e-3,
                max_t_lambda=1.0,
                sinusoidal_dim=512,
                hidden_dim=512,
                output_dim=512,
            ),
        ).get_model()

    def forward(
        self,
        x_t: Float[Tensor, "batch 2"],
        t: Float[Tensor, "batch 1"],
    ) -> Float[Tensor, "batch 2"]:
        return self.velocity(x_t, t=t)

    def get_optimizer(self) -> torch.optim.Optimizer:
        return torch.optim.AdamW(
            self.parameters(),
            lr=experiment_config.lr,
            weight_decay=experiment_config.weight_decay,
        )


class FinalPredictionInverseDecoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.network = StackedResidualMLPConfig.initialize(
            layer_dims=[2, 512, 512, 512, 512, 2],
        ).get_model()

    def forward(
        self,
        x: Float[Tensor, "batch 2"],
    ) -> Float[Tensor, "batch 2"]:
        return self.network(x)

    def get_optimizer(self) -> torch.optim.Optimizer:
        return torch.optim.AdamW(
            self.parameters(),
            lr=experiment_config.inverse_lr,
            weight_decay=experiment_config.inverse_weight_decay,
        )


model = FlowMatchingModel().to(DEVICE)
optimizer = model.get_optimizer()
inverse_decoder = FinalPredictionInverseDecoder().to(DEVICE)
inverse_optimizer = inverse_decoder.get_optimizer()

logs = {
    "steps": [],
    "flow_matching_loss": [],
}
inverse_logs = {
    "steps": [],
    "inverse_loss": [],
}


In [ ]:
FORWARD_COLOR = "#d62728"
BACKWARD_COLOR = "#1f77b4"
LEARNED_INVERSE_COLOR = "#ff7f0e"
MODE_COLORS = ("#1f77b4", "#ff7f0e")
CONTOUR_COLORS = ("#440154", "#3b528b", "#21918c", "#5ec962", "#fde725")
CONTOUR_DASHES = ("solid", "dash", "dot", "longdash")


def current_training_step() -> int:
    if len(logs["steps"]) == 0:
        return 0
    return int(logs["steps"][-1])


def current_inverse_step() -> int:
    if len(inverse_logs["steps"]) == 0:
        return 0
    return int(inverse_logs["steps"][-1])


def append_log_entry(
    *,
    step: int,
    flow_matching_loss: float,
) -> None:
    logs["steps"].append(step)
    logs["flow_matching_loss"].append(flow_matching_loss)


def append_inverse_log_entry(
    *,
    step: int,
    inverse_loss: float,
) -> None:
    inverse_logs["steps"].append(step)
    inverse_logs["inverse_loss"].append(inverse_loss)


def trailing_window(
    *,
    steps: list[int],
    values: list[float],
    window_size: int = 2000,
) -> tuple[list[int], list[float]]:
    start = max(0, len(values) - window_size)
    return steps[start:], values[start:]


def integrate_flow_trajectory(
    *,
    initial_points: Float[Tensor, "batch 2"],
    num_steps: int,
) -> list[Float[Tensor, "batch 2"]]:
    model_was_training = model.training
    model.eval()
    x = initial_points.detach().clone()
    trajectory = [x.detach().clone()]
    dt = 1.0 / float(num_steps)
    with torch.no_grad():
        for step in range(num_steps):
            t = torch.full(
                (x.shape[0], 1),
                fill_value=step / float(num_steps),
                device=x.device,
                dtype=x.dtype,
            )
            with training_device_context():
                velocity = model(x, t)
            x = x + dt * velocity
            trajectory.append(x.detach().clone())
    if model_was_training:
        model.train()
    return trajectory


def integrate_backward_flow_trajectory(
    *,
    final_points: Float[Tensor, "batch 2"],
    num_steps: int,
) -> list[Float[Tensor, "batch 2"]]:
    model_was_training = model.training
    model.eval()
    x = final_points.detach().clone()
    reverse_trajectory = [x.detach().clone()]
    dt = 1.0 / float(num_steps)
    with torch.no_grad():
        for step in range(num_steps, 0, -1):
            t = torch.full(
                (x.shape[0], 1),
                fill_value=step / float(num_steps),
                device=x.device,
                dtype=x.dtype,
            )
            with training_device_context():
                velocity = model(x, t)
            x = x - dt * velocity
            reverse_trajectory.append(x.detach().clone())
    if model_was_training:
        model.train()
    return list(reversed(reverse_trajectory))


def integrate_flow(
    *,
    initial_points: Float[Tensor, "batch 2"],
    num_steps: int,
) -> Float[Tensor, "batch 2"]:
    return integrate_flow_trajectory(
        initial_points=initial_points,
        num_steps=num_steps,
    )[-1]


def integrate_backward_flow(
    *,
    final_points: Float[Tensor, "batch 2"],
    num_steps: int,
) -> Float[Tensor, "batch 2"]:
    return integrate_backward_flow_trajectory(
        final_points=final_points,
        num_steps=num_steps,
    )[0]


def apply_learned_inverse(
    *,
    points: Float[Tensor, "batch 2"],
) -> Float[Tensor, "batch 2"]:
    inverse_was_training = inverse_decoder.training
    inverse_decoder.eval()
    with torch.no_grad():
        decoded = inverse_decoder(points)
    if inverse_was_training:
        inverse_decoder.train()
    return decoded


def axis_limits(
    *,
    points: Float[Tensor, "batch 2"],
    padding: float = 0.10,
) -> tuple[list[float], list[float]]:
    mins = points.min(dim=0).values
    maxs = points.max(dim=0).values
    center = 0.5 * (mins + maxs)
    radius = 0.5 * float((maxs - mins).max().item())
    radius = max(radius * (1.0 + padding), 1.0)
    return (
        [float(center[0] - radius), float(center[0] + radius)],
        [float(center[1] - radius), float(center[1] + radius)],
    )


def plot_training_loss() -> go.Figure:
    step = current_training_step()
    xs, ys = trailing_window(steps=logs["steps"], values=logs["flow_matching_loss"])
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=xs,
            y=ys,
            mode="lines",
            line={"width": 3, "color": FORWARD_COLOR},
            name="flow matching",
        )
    )
    fig.update_layout(
        title=f"flow-matching loss at step {step}",
        height=360,
        width=960,
        margin={"l": 50, "r": 20, "t": 75, "b": 45},
    )
    fig.update_xaxes(title_text="optimization step", showgrid=True, zeroline=False)
    fig.update_yaxes(title_text="loss", showgrid=True, zeroline=False)
    return fig


def plot_inverse_training_loss() -> go.Figure:
    step = current_inverse_step()
    xs, ys = trailing_window(
        steps=inverse_logs["steps"],
        values=inverse_logs["inverse_loss"],
    )
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=xs,
            y=ys,
            mode="lines",
            line={"width": 3, "color": LEARNED_INVERSE_COLOR},
            name="inverse decoder",
        )
    )
    fig.update_layout(
        title=f"inverse decoder loss at step {step}",
        height=360,
        width=960,
        margin={"l": 50, "r": 20, "t": 75, "b": 45},
    )
    fig.update_xaxes(title_text="optimization step", showgrid=True, zeroline=False)
    fig.update_yaxes(title_text="loss", showgrid=True, zeroline=False)
    return fig


def plot_flow_snapshot(
    *,
    scatter_num_samples: int,
) -> go.Figure:
    step = current_training_step()
    with torch.no_grad():
        x_data, _ = data_config.sample_with_mode_ids(scatter_num_samples)
        x_data = x_data.to(DEVICE)
        prior = data_config.sample_prior(scatter_num_samples).to(DEVICE)
        latent_contours, latent_contour_ids, contour_levels = data_config.sample_latent_contours(
            per_contour_size=experiment_config.contour_points_per_level,
            zscores=experiment_config.contour_levels,
        )
        latent_contours = latent_contours.to(DEVICE)
        pushed_samples = integrate_flow(
            initial_points=prior,
            num_steps=experiment_config.integration_steps,
        )
        pushed_contours = integrate_flow(
            initial_points=latent_contours,
            num_steps=experiment_config.integration_steps,
        )
        backward_samples = integrate_backward_flow(
            final_points=x_data,
            num_steps=experiment_config.integration_steps,
        )
        backward_contours = integrate_backward_flow(
            final_points=pushed_contours,
            num_steps=experiment_config.integration_steps,
        )

    forward_points = torch.cat([x_data, pushed_samples, pushed_contours], dim=0)
    backward_points = torch.cat(
        [prior, latent_contours, backward_samples, backward_contours],
        dim=0,
    )
    forward_x_range, forward_y_range = axis_limits(points=forward_points.detach())
    backward_x_range, backward_y_range = axis_limits(points=backward_points.detach())

    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=[
            "forward free-form pushes",
            "forward contour pushes",
            "backward free-form pushes",
            "backward contour pushes",
        ],
        horizontal_spacing=0.07,
        vertical_spacing=0.12,
    )

    fig.add_trace(
        go.Scatter(
            x=x_data[:, 0].detach().cpu().numpy(),
            y=x_data[:, 1].detach().cpu().numpy(),
            mode="markers",
            marker={"size": 4, "color": "#202020", "opacity": 0.18},
            name="data",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=pushed_samples[:, 0].detach().cpu().numpy(),
            y=pushed_samples[:, 1].detach().cpu().numpy(),
            mode="markers",
            marker={"size": 4, "color": FORWARD_COLOR, "opacity": 0.46},
            name="pushed noise",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=x_data[:, 0].detach().cpu().numpy(),
            y=x_data[:, 1].detach().cpu().numpy(),
            mode="markers",
            marker={"size": 4, "color": "#202020", "opacity": 0.10},
            name="data backdrop",
        ),
        row=1,
        col=2,
    )
    for contour_id, contour_level in enumerate(contour_levels):
        mask = latent_contour_ids == contour_id
        fig.add_trace(
            go.Scatter(
                x=pushed_contours[mask, 0].detach().cpu().numpy(),
                y=pushed_contours[mask, 1].detach().cpu().numpy(),
                mode="lines",
                line={
                    "width": 2.5,
                    "color": CONTOUR_COLORS[contour_id % len(CONTOUR_COLORS)],
                },
                name=f"z = {contour_level}",
            ),
            row=1,
            col=2,
        )

    fig.add_trace(
        go.Scatter(
            x=prior[:, 0].detach().cpu().numpy(),
            y=prior[:, 1].detach().cpu().numpy(),
            mode="markers",
            marker={"size": 4, "color": "#202020", "opacity": 0.18},
            name="latent prior",
        ),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=backward_samples[:, 0].detach().cpu().numpy(),
            y=backward_samples[:, 1].detach().cpu().numpy(),
            mode="markers",
            marker={"size": 4, "color": BACKWARD_COLOR, "opacity": 0.46},
            name="actual backward(data)",
        ),
        row=2,
        col=1,
    )

    for contour_id, contour_level in enumerate(contour_levels):
        mask = latent_contour_ids == contour_id
        fig.add_trace(
            go.Scatter(
                x=latent_contours[mask, 0].detach().cpu().numpy(),
                y=latent_contours[mask, 1].detach().cpu().numpy(),
                mode="lines",
                line={
                    "width": 1.7,
                    "color": CONTOUR_COLORS[contour_id % len(CONTOUR_COLORS)],
                    "dash": "dot",
                },
                name=f"latent ref z = {contour_level}",
            ),
            row=2,
            col=2,
        )
        fig.add_trace(
            go.Scatter(
                x=backward_contours[mask, 0].detach().cpu().numpy(),
                y=backward_contours[mask, 1].detach().cpu().numpy(),
                mode="lines",
                line={
                    "width": 2.5,
                    "color": CONTOUR_COLORS[contour_id % len(CONTOUR_COLORS)],
                },
                name=f"actual backward z = {contour_level}",
            ),
            row=2,
            col=2,
        )

    for col, axis_name in [(1, "y"), (2, "y2")]:
        fig.update_xaxes(
            range=forward_x_range,
            showgrid=True,
            zeroline=False,
            scaleanchor=axis_name,
            scaleratio=1,
            row=1,
            col=col,
        )
        fig.update_yaxes(
            range=forward_y_range,
            showgrid=True,
            zeroline=False,
            row=1,
            col=col,
        )
    for col, axis_name in [(1, "y3"), (2, "y4")]:
        fig.update_xaxes(
            range=backward_x_range,
            showgrid=True,
            zeroline=False,
            scaleanchor=axis_name,
            scaleratio=1,
            row=2,
            col=col,
        )
        fig.update_yaxes(
            range=backward_y_range,
            showgrid=True,
            zeroline=False,
            row=2,
            col=col,
        )

    fig.update_layout(
        title=f"flow-matching snapshot at step {step}",
        height=860,
        width=1350,
        margin={"l": 30, "r": 20, "t": 85, "b": 30},
        legend={"x": 1.02, "xanchor": "left", "y": 1.0, "yanchor": "top"},
    )
    return fig


def build_noise_toggle_plot(
    *,
    scatter_num_samples: int,
) -> go.Figure:
    step = current_training_step()
    with torch.no_grad():
        x_data, _ = data_config.sample_with_mode_ids(scatter_num_samples)
        x_data = x_data.to(DEVICE)
        prior = data_config.sample_prior(scatter_num_samples).to(DEVICE)
        latent_contours, latent_contour_ids, contour_levels = data_config.sample_latent_contours(
            per_contour_size=experiment_config.contour_points_per_level,
            zscores=experiment_config.contour_levels,
        )
        latent_contours = latent_contours.to(DEVICE)
        forward_sample_trajectory = integrate_flow_trajectory(
            initial_points=prior,
            num_steps=experiment_config.integration_steps,
        )
        forward_contour_trajectory = integrate_flow_trajectory(
            initial_points=latent_contours,
            num_steps=experiment_config.integration_steps,
        )
        backward_sample_trajectory = integrate_backward_flow_trajectory(
            final_points=x_data,
            num_steps=experiment_config.integration_steps,
        )

    snapshot_indices = (
        torch.linspace(
            0,
            experiment_config.integration_steps,
            experiment_config.final_toggle_steps,
        )
        .round()
        .long()
        .tolist()
    )

    forward_points = torch.cat(
        [x_data, *forward_sample_trajectory, *forward_contour_trajectory],
        dim=0,
    )
    backward_points = torch.cat([prior, *backward_sample_trajectory], dim=0)
    forward_x_range, forward_y_range = axis_limits(points=forward_points.detach())
    backward_x_range, backward_y_range = axis_limits(points=backward_points.detach())

    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=[
            "forward free-form pushes",
            "forward contour pushes",
            "actual backward free-form pushes",
            "latent reference contours",
        ],
        horizontal_spacing=0.07,
        vertical_spacing=0.12,
    )

    traces_per_snapshot = []
    buttons = []
    for snapshot_index, trajectory_index in enumerate(snapshot_indices):
        visible = snapshot_index == 0
        start_trace_count = len(fig.data)
        flow_time = trajectory_index / float(experiment_config.integration_steps)
        noise_level = 1.0 - flow_time

        current_forward_samples = forward_sample_trajectory[trajectory_index]
        current_forward_contours = forward_contour_trajectory[trajectory_index]
        current_backward_samples = backward_sample_trajectory[trajectory_index]

        fig.add_trace(
            go.Scatter(
                x=x_data[:, 0].detach().cpu().numpy(),
                y=x_data[:, 1].detach().cpu().numpy(),
                mode="markers",
                marker={"size": 4, "color": "#202020", "opacity": 0.14},
                name="data",
                visible=visible,
                showlegend=snapshot_index == 0,
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=current_forward_samples[:, 0].detach().cpu().numpy(),
                y=current_forward_samples[:, 1].detach().cpu().numpy(),
                mode="markers",
                marker={"size": 4, "color": FORWARD_COLOR, "opacity": 0.48},
                name="pushed noise",
                visible=visible,
                showlegend=snapshot_index == 0,
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=x_data[:, 0].detach().cpu().numpy(),
                y=x_data[:, 1].detach().cpu().numpy(),
                mode="markers",
                marker={"size": 4, "color": "#202020", "opacity": 0.10},
                name="data backdrop",
                visible=visible,
                showlegend=False,
            ),
            row=1,
            col=2,
        )
        for contour_id, contour_level in enumerate(contour_levels):
            mask = latent_contour_ids == contour_id
            fig.add_trace(
                go.Scatter(
                    x=current_forward_contours[mask, 0].detach().cpu().numpy(),
                    y=current_forward_contours[mask, 1].detach().cpu().numpy(),
                    mode="lines",
                    line={
                        "width": 2.5,
                        "color": CONTOUR_COLORS[contour_id % len(CONTOUR_COLORS)],
                    },
                    name=f"forward z = {contour_level}",
                    visible=visible,
                    showlegend=snapshot_index == 0,
                ),
                row=1,
                col=2,
            )

        fig.add_trace(
            go.Scatter(
                x=prior[:, 0].detach().cpu().numpy(),
                y=prior[:, 1].detach().cpu().numpy(),
                mode="markers",
                marker={"size": 4, "color": "#202020", "opacity": 0.14},
                name="latent prior",
                visible=visible,
                showlegend=snapshot_index == 0,
            ),
            row=2,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=current_backward_samples[:, 0].detach().cpu().numpy(),
                y=current_backward_samples[:, 1].detach().cpu().numpy(),
                mode="markers",
                marker={"size": 4, "color": BACKWARD_COLOR, "opacity": 0.48},
                name="actual backward(data)",
                visible=visible,
                showlegend=snapshot_index == 0,
            ),
            row=2,
            col=1,
        )
        for contour_id, contour_level in enumerate(contour_levels):
            mask = latent_contour_ids == contour_id
            fig.add_trace(
                go.Scatter(
                    x=latent_contours[mask, 0].detach().cpu().numpy(),
                    y=latent_contours[mask, 1].detach().cpu().numpy(),
                    mode="lines",
                    line={
                        "width": 1.7,
                        "color": CONTOUR_COLORS[contour_id % len(CONTOUR_COLORS)],
                        "dash": "dot",
                    },
                    name=f"latent ref z = {contour_level}",
                    visible=visible,
                    showlegend=snapshot_index == 0,
                ),
                row=2,
                col=2,
            )

        traces_per_snapshot.append(len(fig.data) - start_trace_count)
        buttons.append(
            {
                "label": f"noise={noise_level:.2f}",
                "method": "update",
                "args": [
                    {
                        "visible": (
                            [False] * sum(traces_per_snapshot)
                        ),
                    },
                    {
                        "title": (
                            "noise-to-data flow at step "
                            f"{step} for remaining noise={noise_level:.2f}"
                            f" (t={flow_time:.2f})"
                        )
                    },
                ],
            }
        )
        start = sum(traces_per_snapshot[:-1])
        end = start + traces_per_snapshot[-1]
        visibility = [False] * sum(traces_per_snapshot)
        for trace_index in range(start, end):
            visibility[trace_index] = True
        buttons[-1]["args"][0]["visible"] = visibility

    for col, axis_name in [(1, "y"), (2, "y2")]:
        fig.update_xaxes(
            range=forward_x_range,
            showgrid=True,
            zeroline=False,
            scaleanchor=axis_name,
            scaleratio=1,
            row=1,
            col=col,
        )
        fig.update_yaxes(
            range=forward_y_range,
            showgrid=True,
            zeroline=False,
            row=1,
            col=col,
        )
    for col, axis_name in [(1, "y3"), (2, "y4")]:
        fig.update_xaxes(
            range=backward_x_range,
            showgrid=True,
            zeroline=False,
            scaleanchor=axis_name,
            scaleratio=1,
            row=2,
            col=col,
        )
        fig.update_yaxes(
            range=backward_y_range,
            showgrid=True,
            zeroline=False,
            row=2,
            col=col,
        )

    first_flow_time = snapshot_indices[0] / float(experiment_config.integration_steps)
    first_noise_level = 1.0 - first_flow_time
    fig.update_layout(
        title=(
            "noise-to-data flow at step "
            f"{step} for remaining noise={first_noise_level:.2f}"
            f" (t={first_flow_time:.2f})"
        ),
        height=860,
        width=1350,
        margin={"l": 30, "r": 20, "t": 100, "b": 30},
        legend={"x": 1.02, "xanchor": "left", "y": 1.0, "yanchor": "top"},
        updatemenus=[
            {
                "type": "dropdown",
                "direction": "down",
                "buttons": buttons,
                "x": 0.0,
                "xanchor": "left",
                "y": 1.12,
                "yanchor": "top",
                "showactive": True,
            }
        ],
        annotations=[
            {
                "text": "remaining noise",
                "x": 0.0,
                "xanchor": "left",
                "y": 1.18,
                "yanchor": "top",
                "xref": "paper",
                "yref": "paper",
                "showarrow": False,
                "font": {"size": 13},
            }
        ],
    )
    return fig


def build_inverse_training_dataset() -> tuple[Tensor, Tensor]:
    num_batches = experiment_config.inverse_dataset_size // experiment_config.inverse_batch_size
    predicted_batches = []
    latent_batches = []
    for _ in tqdm(range(num_batches), desc="building inverse dataset"):
        latent_batch = data_config.sample_prior(experiment_config.inverse_batch_size).to(DEVICE)
        predicted_batch = integrate_flow(
            initial_points=latent_batch,
            num_steps=experiment_config.integration_steps,
        )
        predicted_batches.append(predicted_batch.detach())
        latent_batches.append(latent_batch.detach())
    return torch.cat(predicted_batches, dim=0), torch.cat(latent_batches, dim=0)


def plot_inverse_decoder_snapshot(
    *,
    scatter_num_samples: int,
) -> go.Figure:
    flow_step = current_training_step()
    inverse_step = current_inverse_step()
    with torch.no_grad():
        x_data, _ = data_config.sample_with_mode_ids(scatter_num_samples)
        x_data = x_data.to(DEVICE)
        prior = data_config.sample_prior(scatter_num_samples).to(DEVICE)
        latent_contours, latent_contour_ids, contour_levels = data_config.sample_latent_contours(
            per_contour_size=experiment_config.contour_points_per_level,
            zscores=experiment_config.contour_levels,
        )
        data_mode_contours, data_mode_contour_ids, data_mode_ids, _ = (
            data_config.sample_data_mode_contours(
                per_contour_size=experiment_config.contour_points_per_level,
                zscores=experiment_config.contour_levels,
            )
        )
        latent_contours = latent_contours.to(DEVICE)
        data_mode_contours = data_mode_contours.to(DEVICE)
        predicted_samples = integrate_flow(
            initial_points=prior,
            num_steps=experiment_config.integration_steps,
        )
        predicted_contours = integrate_flow(
            initial_points=latent_contours,
            num_steps=experiment_config.integration_steps,
        )
        actual_backward_samples = integrate_backward_flow(
            final_points=predicted_samples,
            num_steps=experiment_config.integration_steps,
        )
        learned_backward_samples = apply_learned_inverse(points=predicted_samples)
        actual_mode_backward_contours = integrate_backward_flow(
            final_points=data_mode_contours,
            num_steps=experiment_config.integration_steps,
        )
        learned_mode_backward_contours = apply_learned_inverse(points=data_mode_contours)

    forward_points = torch.cat([x_data, predicted_samples, predicted_contours], dim=0)
    backward_points = torch.cat(
        [
            prior,
            latent_contours,
            actual_backward_samples,
            learned_backward_samples,
            actual_mode_backward_contours,
            learned_mode_backward_contours,
        ],
        dim=0,
    )
    forward_x_range, forward_y_range = axis_limits(points=forward_points.detach())
    backward_x_range, backward_y_range = axis_limits(points=backward_points.detach())

    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=[
            "model final predictions",
            "forward contour pushes",
            "backward free-form: integrated vs learned",
            "two-mode backward contours under learned inverse",
        ],
        horizontal_spacing=0.07,
        vertical_spacing=0.12,
    )

    fig.add_trace(
        go.Scatter(
            x=x_data[:, 0].detach().cpu().numpy(),
            y=x_data[:, 1].detach().cpu().numpy(),
            mode="markers",
            marker={"size": 4, "color": "#202020", "opacity": 0.16},
            name="data",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=predicted_samples[:, 0].detach().cpu().numpy(),
            y=predicted_samples[:, 1].detach().cpu().numpy(),
            mode="markers",
            marker={"size": 4, "color": FORWARD_COLOR, "opacity": 0.44},
            name="model final prediction",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=x_data[:, 0].detach().cpu().numpy(),
            y=x_data[:, 1].detach().cpu().numpy(),
            mode="markers",
            marker={"size": 4, "color": "#202020", "opacity": 0.10},
            name="data backdrop",
        ),
        row=1,
        col=2,
    )
    for contour_id, contour_level in enumerate(contour_levels):
        mask = latent_contour_ids == contour_id
        fig.add_trace(
            go.Scatter(
                x=predicted_contours[mask, 0].detach().cpu().numpy(),
                y=predicted_contours[mask, 1].detach().cpu().numpy(),
                mode="lines",
                line={
                    "width": 2.5,
                    "color": CONTOUR_COLORS[contour_id % len(CONTOUR_COLORS)],
                },
                name=f"forward z = {contour_level}",
            ),
            row=1,
            col=2,
        )

    fig.add_trace(
        go.Scatter(
            x=prior[:, 0].detach().cpu().numpy(),
            y=prior[:, 1].detach().cpu().numpy(),
            mode="markers",
            marker={"size": 4, "color": "#202020", "opacity": 0.14},
            name="latent prior",
        ),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=actual_backward_samples[:, 0].detach().cpu().numpy(),
            y=actual_backward_samples[:, 1].detach().cpu().numpy(),
            mode="markers",
            marker={"size": 4, "color": BACKWARD_COLOR, "opacity": 0.42},
            name="actual integrated backward",
        ),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=learned_backward_samples[:, 0].detach().cpu().numpy(),
            y=learned_backward_samples[:, 1].detach().cpu().numpy(),
            mode="markers",
            marker={"size": 4, "color": LEARNED_INVERSE_COLOR, "opacity": 0.42},
            name="learned inverse decoder",
        ),
        row=2,
        col=1,
    )

    for contour_id, contour_level in enumerate(contour_levels):
        latent_mask = latent_contour_ids == contour_id
        fig.add_trace(
            go.Scatter(
                x=latent_contours[latent_mask, 0].detach().cpu().numpy(),
                y=latent_contours[latent_mask, 1].detach().cpu().numpy(),
                mode="lines",
                line={
                    "width": 1.5,
                    "color": "#444444",
                    "dash": "dot",
                },
                name=f"latent ref z = {contour_level}",
            ),
            row=2,
            col=2,
        )
        actual_mask = data_mode_contour_ids == contour_id
        fig.add_trace(
            go.Scatter(
                x=actual_mode_backward_contours[actual_mask, 0].detach().cpu().numpy(),
                y=actual_mode_backward_contours[actual_mask, 1].detach().cpu().numpy(),
                mode="lines",
                line={
                    "width": 1.7,
                    "color": "#111111",
                    "dash": CONTOUR_DASHES[contour_id % len(CONTOUR_DASHES)],
                },
                name=f"actual backward z = {contour_level}",
            ),
            row=2,
            col=2,
        )
    for mode_id, color in enumerate(MODE_COLORS):
        for contour_id, contour_level in enumerate(contour_levels):
            learned_mask = (data_mode_ids == mode_id) & (data_mode_contour_ids == contour_id)
            fig.add_trace(
                go.Scatter(
                    x=learned_mode_backward_contours[learned_mask, 0].detach().cpu().numpy(),
                    y=learned_mode_backward_contours[learned_mask, 1].detach().cpu().numpy(),
                    mode="lines",
                    line={
                        "width": 2.5,
                        "color": color,
                        "dash": CONTOUR_DASHES[contour_id % len(CONTOUR_DASHES)],
                    },
                    name=f"learned inverse mode {mode_id} z = {contour_level}",
                ),
                row=2,
                col=2,
            )

    for col, axis_name in [(1, "y"), (2, "y2")]:
        fig.update_xaxes(
            range=forward_x_range,
            showgrid=True,
            zeroline=False,
            scaleanchor=axis_name,
            scaleratio=1,
            row=1,
            col=col,
        )
        fig.update_yaxes(
            range=forward_y_range,
            showgrid=True,
            zeroline=False,
            row=1,
            col=col,
        )
    for col, axis_name in [(1, "y3"), (2, "y4")]:
        fig.update_xaxes(
            range=backward_x_range,
            showgrid=True,
            zeroline=False,
            scaleanchor=axis_name,
            scaleratio=1,
            row=2,
            col=col,
        )
        fig.update_yaxes(
            range=backward_y_range,
            showgrid=True,
            zeroline=False,
            row=2,
            col=col,
        )

    fig.update_layout(
        title=(
            "inverse decoder diagnostics after flow training step "
            f"{flow_step} and inverse step {inverse_step}"
        ),
        height=860,
        width=1350,
        margin={"l": 30, "r": 20, "t": 85, "b": 30},
        legend={"x": 1.02, "xanchor": "left", "y": 1.0, "yanchor": "top"},
    )
    return fig


In [ ]:
for step in tqdm(range(1, experiment_config.total_steps + 1), desc="flow training"):
    optimizer.zero_grad(set_to_none=True)
    x_target = data_config.sample(experiment_config.train_batch_size).to(DEVICE)
    x_noise = data_config.sample_prior(experiment_config.train_batch_size).to(DEVICE)
    t = torch.rand((experiment_config.train_batch_size, 1), device=DEVICE)
    x_t = (1.0 - t) * x_noise + t * x_target
    target_velocity = x_target - x_noise

    with training_device_context():
        predicted_velocity = model(x_t, t)
        flow_matching_loss = F.huber_loss(
            predicted_velocity.float(),
            target_velocity.float(),
            delta=3.0,
        )

    flow_matching_loss.backward()
    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=experiment_config.grad_clip_norm,
    )
    optimizer.step()

    should_log = (
        step == 1
        or step % experiment_config.log_every_steps == 0
        or step % experiment_config.snapshot_every_steps == 0
        or step == experiment_config.total_steps
    )
    if should_log:
        append_log_entry(
            step=step,
            flow_matching_loss=float(flow_matching_loss.detach().item()),
        )

    if (
        step == 1
        or step % experiment_config.snapshot_every_steps == 0
        or step == experiment_config.total_steps
    ):
        plot_flow_snapshot(
            scatter_num_samples=experiment_config.scatter_num_samples,
        ).show()


plot_training_loss().show()
build_noise_toggle_plot(
    scatter_num_samples=experiment_config.scatter_num_samples,
).show()

inverse_dataset_predictions, inverse_dataset_latents = build_inverse_training_dataset()
dataset_size = int(inverse_dataset_predictions.shape[0])

for inverse_step in tqdm(
    range(1, experiment_config.inverse_steps + 1),
    desc="inverse decoder training",
):
    inverse_optimizer.zero_grad(set_to_none=True)
    batch_indices = torch.randint(
        low=0,
        high=dataset_size,
        size=(experiment_config.inverse_batch_size,),
        device=DEVICE,
    )
    x_batch = inverse_dataset_predictions[batch_indices]
    z_batch = inverse_dataset_latents[batch_indices]
    with training_device_context():
        z_pred = inverse_decoder(x_batch)
        inverse_loss = F.huber_loss(
            z_pred.float(),
            z_batch.float(),
            delta=2.0,
        )
    inverse_loss.backward()
    torch.nn.utils.clip_grad_norm_(
        inverse_decoder.parameters(),
        max_norm=experiment_config.grad_clip_norm,
    )
    inverse_optimizer.step()

    if (
        inverse_step == 1
        or inverse_step % experiment_config.inverse_log_every_steps == 0
        or inverse_step == experiment_config.inverse_steps
    ):
        append_inverse_log_entry(
            step=inverse_step,
            inverse_loss=float(inverse_loss.detach().item()),
        )


plot_inverse_training_loss().show()
plot_inverse_decoder_snapshot(
    scatter_num_samples=experiment_config.scatter_num_samples,
).show()
